In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, datasets, models
from torch.utils.data.dataloader import DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn import preprocessing
import numpy as np
import copy
import os
import time

# PART 0: LARS Optimizer Implementation
class LARS(torch.optim.Optimizer):
    def __init__(self, params, lr, momentum=0.9, weight_decay=1e-4, trust_coefficient=0.001, eps=1e-8):
        defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay,
                        trust_coefficient=trust_coefficient, eps=eps)
        super(LARS, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            weight_decay = group['weight_decay']
            momentum = group['momentum']
            trust_coefficient = group['trust_coefficient']
            eps = group['eps']

            for p in group['params']:
                if p.grad is None:
                    continue
                d_p = p.grad

                p_norm = torch.norm(p.data)
                g_norm = torch.norm(d_p.data)

                if p_norm > 0 and g_norm > 0:
                    lars_lr = trust_coefficient * p_norm / (g_norm + p_norm * weight_decay + eps)
                    d_p = d_p.add(p, alpha=weight_decay)
                    d_p *= lars_lr

                param_state = self.state[p]
                if 'momentum_buffer' not in param_state:
                    buf = param_state['momentum_buffer'] = torch.clone(d_p).detach()
                else:
                    buf = param_state['momentum_buffer']
                    buf.mul_(momentum).add_(d_p)

                p.add_(buf, alpha=-group['lr'])
        return loss

# PART 1: Data Augmentation
class GaussianBlur(object):
    def __init__(self, kernel_size):
        radius = kernel_size // 2
        kernel_size = radius * 2 + 1
        self.blur_h = nn.Conv2d(3, 3, kernel_size=(kernel_size, 1), stride=1, padding=0, bias=False, groups=3)
        self.blur_v = nn.Conv2d(3, 3, kernel_size=(1, kernel_size), stride=1, padding=0, bias=False, groups=3)
        self.k, self.r = kernel_size, radius
        self.blur = nn.Sequential(nn.ReflectionPad2d(radius), self.blur_h, self.blur_v)
        self.pil_to_tensor = transforms.ToTensor()
        self.tensor_to_pil = transforms.ToPILImage()

    def __call__(self, img):
        img = self.pil_to_tensor(img).unsqueeze(0)
        sigma = np.random.uniform(0.1, 2.0)
        x = np.arange(-self.r, self.r + 1)
        x = np.exp(-np.power(x, 2) / (2 * sigma * sigma))
        x = x / x.sum()
        x = torch.from_numpy(x).view(1, -1).repeat(3, 1).float()
        self.blur_h.weight.data.copy_(x.view(3, 1, self.k, 1))
        self.blur_v.weight.data.copy_(x.view(3, 1, 1, self.k))
        with torch.no_grad():
            img = self.blur(img).squeeze()
        return self.tensor_to_pil(img)

class MultiViewDataInjector(object):
    def __init__(self, transform_list):
        self.transforms = transform_list
    def __call__(self, sample):
        return [t(sample) for t in self.transforms]

def get_simclr_data_transforms(input_shape, s=1):
    size = input_shape[0]
    color_jitter = transforms.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)
    return transforms.Compose([
        transforms.RandomResizedCrop(size=size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([color_jitter], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        GaussianBlur(kernel_size=int(0.1 * size)),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])

# PART 2: Model Architectures
class MLPHead(nn.Module):
    def __init__(self, in_channels, mlp_hidden_size, projection_size):
        super(MLPHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_channels, mlp_hidden_size),
            nn.BatchNorm1d(mlp_hidden_size),
            nn.ReLU(inplace=True),
            nn.Linear(mlp_hidden_size, projection_size)
        )
    def forward(self, x):
        return self.net(x)

class ResNetEncoder(torch.nn.Module):
    def __init__(self, name, projection_head):
        super(ResNetEncoder, self).__init__()
        resnet = models.resnet50(weights=None) if name == 'resnet50' else models.resnet18(weights=None)
        self.encoder = torch.nn.Sequential(*list(resnet.children())[:-1])
        self.projection = MLPHead(in_channels=resnet.fc.in_features, **projection_head)

    def forward(self, x):
        h = self.encoder(x)
        h = h.view(h.shape[0], -1)
        return self.projection(h)

# PART 3: Original BYOL Trainer
class BYOLTrainerRefined:
    def __init__(self, online_net, target_net, predictor, optimizer, device, **params):
        self.online_net = online_net
        self.target_net = target_net
        self.predictor = predictor
        self.optimizer = optimizer
        self.device = device
        self.max_epochs, self.m = params['max_epochs'], params['m']
        self.batch_size, self.num_workers = params['batch_size'], params['num_workers']
        self.writer = SummaryWriter()

    def _update_target(self):
        for p_o, p_t in zip(self.online_net.parameters(), self.target_net.parameters()):
            p_t.data = p_t.data * self.m + p_o.data * (1. - self.m)

    def regression_loss(self, x, y):
        x, y = F.normalize(x, dim=1), F.normalize(y, dim=1)
        return 2 - 2 * (x * y).sum(dim=-1)

    def train(self, train_dataset):
        loader = DataLoader(train_dataset, batch_size=self.batch_size, num_workers=self.num_workers, drop_last=True, shuffle=True)
        for epoch in range(self.max_epochs):
            for i, ((view_1, view_2), _) in enumerate(loader):
                view_1, view_2 = view_1.to(self.device), view_2.to(self.device)

                # Online predictions
                p1 = self.predictor(self.online_net(view_1))
                p2 = self.predictor(self.online_net(view_2))

                # Target projections (Stop Gradient)
                with torch.no_grad():
                    t1 = self.target_net(view_2)
                    t2 = self.target_net(view_1)

                # Symmetric BYOL Loss
                loss = (self.regression_loss(p1, t1) + self.regression_loss(p2, t2)).mean()

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                self._update_target()

                if i % 10 == 0:
                    print(f"Epoch {epoch}, Iter {i}, BYOL Loss (LARS): {loss.item():.4f}")

        save_path = os.path.join(self.writer.log_dir, 'original_byol_lars.pth')
        torch.save({'online_state': self.online_net.state_dict()}, save_path)
        print(f"Weights saved to {save_path}")
        return save_path

# PART 4: Linear Evaluation
def run_linear_evaluation(encoder, train_loader, test_loader, device):
    print("\n--- Linear Evaluation (Frozen ResNet-50) ---")
    encoder.eval()

    def extract_features(model, loader):
        x_out, y_out = [], []
        for img, lbl in loader:
            with torch.no_grad():
                img = img.to(device)
                f = model(img).view(img.size(0), -1).cpu()
                x_out.append(f); y_out.extend(lbl.numpy())
        return torch.cat(x_out, dim=0), torch.tensor(y_out)

    x_train, y_train = extract_features(encoder, train_loader)
    x_test, y_test = extract_features(encoder, test_loader)

    scaler = preprocessing.StandardScaler()
    x_train_scaled = torch.from_numpy(scaler.fit_transform(x_train)).float()
    x_test_scaled = torch.from_numpy(scaler.transform(x_test)).float()

    loader = DataLoader(torch.utils.data.TensorDataset(x_train_scaled, y_train), batch_size=64, shuffle=True)
    classifier = torch.nn.Linear(x_train_scaled.shape[1], 10).to(device)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=3e-4)

    for epoch in range(100):
        for x_b, y_b in loader:
            optimizer.zero_grad()
            F.cross_entropy(classifier(x_b.to(device)), y_b.to(device)).backward()
            optimizer.step()

        if epoch % 10 == 0:
            with torch.no_grad():
                preds = torch.argmax(classifier(x_test_scaled.to(device)), dim=1)
                acc = (preds == y_test.to(device)).float().mean()
                print(f"Epoch {epoch} | Accuracy: {acc.item()*100:.2f}%")

# PART 5: Execution
if __name__ == '__main__':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 1. Initialize Online and Target Networks (ResNet-50)

    online_net = ResNetEncoder('resnet50', {'mlp_hidden_size': 4096, 'projection_size': 256}).to(device)
    target_net = copy.deepcopy(online_net)
    predictor = MLPHead(256, 4096, 256).to(device)

    # Initialize targets with stop-gradient
    for po, pt in zip(online_net.parameters(), target_net.parameters()):
        pt.data.copy_(po.data); pt.requires_grad = False

    # 2. Configure LARS Optimizer with lr=0.02
    optimizer = LARS(
        list(online_net.parameters()) + list(predictor.parameters()),
        lr=0.02,
        momentum=0.9,
        weight_decay=1e-6
    )

    # 3. Data Preparation (STL-10)
    transform = get_simclr_data_transforms((32, 32, 3))
    train_ds = datasets.CIFAR10('./data', train=True, download=True,
                               transform=MultiViewDataInjector([transform, transform]))

    # 4. Pre-training
    trainer = BYOLTrainerRefined(online_net, target_net, predictor, optimizer, device,
                                 max_epochs=10, m=0.996, batch_size=256, num_workers=2)
    trainer.train(train_ds)

    # 5. Final Evaluation
    eval_t = transforms.Compose([transforms.ToTensor()])
    tr_ldr = DataLoader(datasets.CIFAR10('./data', train=True, transform=eval_t), batch_size=256)
    te_ldr = DataLoader(datasets.CIFAR10('./data', train=False, transform=eval_t), batch_size=256)

    run_linear_evaluation(online_net.encoder, tr_ldr, te_ldr, device)

2026-03-23 11:26:52.313419: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774265212.537624      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774265212.608015      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774265213.133340      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774265213.133394      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774265213.133397      55 computation_placer.cc:177] computation placer alr

Epoch 0, Iter 0, BYOL Loss (LARS): 4.0125
Epoch 0, Iter 10, BYOL Loss (LARS): 3.8638
Epoch 0, Iter 20, BYOL Loss (LARS): 3.6181
Epoch 0, Iter 30, BYOL Loss (LARS): 3.3423
Epoch 0, Iter 40, BYOL Loss (LARS): 3.0659
Epoch 0, Iter 50, BYOL Loss (LARS): 2.8397
Epoch 0, Iter 60, BYOL Loss (LARS): 2.6343
Epoch 0, Iter 70, BYOL Loss (LARS): 2.4342
Epoch 0, Iter 80, BYOL Loss (LARS): 2.3122
Epoch 0, Iter 90, BYOL Loss (LARS): 2.2177
Epoch 0, Iter 100, BYOL Loss (LARS): 2.1400
Epoch 0, Iter 110, BYOL Loss (LARS): 2.0961
Epoch 0, Iter 120, BYOL Loss (LARS): 2.0131
Epoch 0, Iter 130, BYOL Loss (LARS): 1.9593
Epoch 0, Iter 140, BYOL Loss (LARS): 1.9379
Epoch 0, Iter 150, BYOL Loss (LARS): 1.8890
Epoch 0, Iter 160, BYOL Loss (LARS): 1.8696
Epoch 0, Iter 170, BYOL Loss (LARS): 1.8414
Epoch 0, Iter 180, BYOL Loss (LARS): 1.8153
Epoch 0, Iter 190, BYOL Loss (LARS): 1.8120
Epoch 1, Iter 0, BYOL Loss (LARS): 1.8058
Epoch 1, Iter 10, BYOL Loss (LARS): 1.7953
Epoch 1, Iter 20, BYOL Loss (LARS): 1.7843
Epo